# PolyMerge: Universal Architecture-Agnostic LLM Merger & Toolkit

Interactive Studio for **Google Colab** and **Kaggle**.

### Supported Capabilities:
1. **Core Model Merging**: 12 Methods (`linear`, `slerp`, `nuslerp`, `karcher`, `nearswap`, `task_arithmetic`, `ties`, `dare_ties`, `della`, `omnimerge_v2`, `model_stock`, `sce`).
2. **Vision / Multimodal Layer Filtering**: Automatically ignores vision encoders (`visual.*`, `vision_tower.*`, `mm_projector.*`).
3. **MoE Upcycling**: Dense-to-MoE assembly (`Mixtral`, `Qwen-MoE`, `DeepSeek-MoE`, `Gemma 4 MoE`).
4. **Layer Slicing & Frankenmerging**: Arbitrary layer duplication and stacking (e.g. 32L -> 48L).
5. **LoRA Toolkit**: SVD extraction, in-place base baking, and multi-adapter combination.
6. **Model Pruning**: Zero-retraining Dense Layer Pruning (Local Residual Healing) & MoE Expert Pruning (T18 Soft-Transfer).
7. **Tokenizer Surgery**: Automatic vocabulary alignment, padding to 64x for Tensor Cores, and truncation.
8. **Sanity Audit & Publishing**: Detect NaN/Inf tensors, router condition degeneracy, and direct push to Hugging Face Hub.

In [ ]:
#@title 1. Environment Setup & Fast Transfer
!pip install -q pyyaml rich huggingface-hub safetensors hf-transfer
!git clone -q https://github.com/ehristoforu/polymerge.git || true
%cd polymerge
!pip install -q -e .
import os
from huggingface_hub import login
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# Optional: login to Hugging Face if pushing models or accessing gated repos
# login(token="hf_...")
print("PolyMerge Environment Ready!")

In [ ]:
#@title 2. Universal Interactive Studio
import ipywidgets as widgets
from IPython.display import display, clear_output

# Tab 1: Model Merging
merge_method = widgets.Dropdown(
    options=['linear', 'slerp', 'nuslerp', 'karcher', 'nearswap', 'task_arithmetic', 'dare_ties', 'della', 'omnimerge_v2', 'model_stock', 'sce', 'passthrough'],
    value='omnimerge_v2',
    description='Method:'
)
base_model_in = widgets.Text(value='Qwen/Qwen2.5-7B', description='Base Model:')
m1_in = widgets.Text(value='Qwen/Qwen2.5-Coder-7B-Instruct', description='Model 1:')
w1_in = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='Weight 1:')
d1_in = widgets.FloatSlider(value=0.6, min=0.1, max=1.0, step=0.05, description='Density 1:')

m2_in = widgets.Text(value='deepseek-ai/DeepSeek-R1-Distill-Qwen-7B', description='Model 2:')
w2_in = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='Weight 2:')
d2_in = widgets.FloatSlider(value=0.6, min=0.1, max=1.0, step=0.05, description='Density 2:')

vision_check = widgets.Checkbox(value=True, description='Ignore Vision/Multimodal Encoders')
skip_mlp_check = widgets.Checkbox(value=True, description='Protect Thinking MLP Attractors (Qwen3.6+)')
merge_out_dir = widgets.Text(value='./merged_model', description='Output Dir:')
merge_btn = widgets.Button(description='Run Model Merge', button_style='primary')

tab_merge = widgets.VBox([
    merge_method, base_model_in,
    m1_in, w1_in, d1_in,
    m2_in, w2_in, d2_in,
    vision_check, skip_mlp_check, merge_out_dir, merge_btn
])

# Tab 2: MoE Upcycling
moe_base = widgets.Text(value='mistralai/Mistral-7B-v0.3', description='Base Model:')
moe_experts = widgets.Textarea(
    value='mistralai/Mistral-7B-Instruct-v0.3\nQwen/Qwen2.5-Coder-7B-Instruct\ndeepseek-ai/DeepSeek-R1-Distill-Qwen-7B',
    description='Experts (1/line):',
    layout=widgets.Layout(height='100px')
)
moe_arch = widgets.Dropdown(options=['mixtral', 'qwen', 'deepseek', 'gemma'], value='mixtral', description='Target MoE:')
moe_gate = widgets.Dropdown(options=['uniform_random', 'random'], value='uniform_random', description='Gate Mode:')
moe_shared = widgets.Text(value='', description='Shared Exp (Opt):')
moe_out_dir = widgets.Text(value='./assembled_moe', description='Output Dir:')
moe_btn = widgets.Button(description='Assemble MoE Model', button_style='success')

tab_moe = widgets.VBox([moe_base, moe_experts, moe_arch, moe_gate, moe_shared, moe_out_dir, moe_btn])

# Tab 3: Model Pruning
prune_type = widgets.Dropdown(options=['Dense Layer Pruning', 'MoE Expert Pruning'], value='Dense Layer Pruning', description='Type:')
prune_model = widgets.Text(value='meta-llama/Llama-3.2-1B', description='Target Model:')
prune_dense_frac = widgets.FloatSlider(value=0.2, min=0.05, max=0.5, step=0.05, description='Drop Frac:')
prune_moe_experts = widgets.IntSlider(value=64, min=8, max=128, step=8, description='Surviving Exp:')
prune_out_dir = widgets.Text(value='./pruned_model', description='Output Dir:')
prune_btn = widgets.Button(description='Execute Pruning', button_style='warning')

tab_prune = widgets.VBox([prune_type, prune_model, prune_dense_frac, prune_moe_experts, prune_out_dir, prune_btn])

# Tab 4: LoRA Tools
lora_action = widgets.Dropdown(options=['Extract LoRA (SVD)', 'Bake LoRA into Base', 'Merge Multiple Adapters'], value='Extract LoRA (SVD)', description='Action:')
lora_base = widgets.Text(value='meta-llama/Llama-3.2-1B', description='Base Model:')
lora_tuned = widgets.Text(value='./finetuned_model', description='Tuned Model:')
lora_adapter = widgets.Text(value='./my_lora', description='Adapter Path:')
lora_rank = widgets.IntSlider(value=16, min=4, max=64, step=4, description='Rank:')
lora_out_dir = widgets.Text(value='./lora_output', description='Output Dir:')
lora_btn = widgets.Button(description='Run LoRA Tool', button_style='info')

tab_lora = widgets.VBox([lora_action, lora_base, lora_tuned, lora_adapter, lora_rank, lora_out_dir, lora_btn])

# Tab 5: Tokenizer Surgery
tok_action = widgets.Dropdown(options=['Align Vocabularies', 'Truncate Vocabulary'], value='Align Vocabularies', description='Action:')
tok_base = widgets.Text(value='meta-llama/Llama-3.2-1B', description='Base Model:')
tok_sources = widgets.Text(value='meta-llama/Llama-3.2-1B-Instruct', description='Sources:')
tok_truncate_n = widgets.IntText(value=128000, description='Target Vocab:')
tok_out_dir = widgets.Text(value='./aligned_model', description='Output Dir:')
tok_btn = widgets.Button(description='Run Tokenizer Surgery', button_style='danger')

tab_tok = widgets.VBox([tok_action, tok_base, tok_sources, tok_truncate_n, tok_out_dir, tok_btn])

# Tab 6: Sanity Audit & Publish
audit_target = widgets.Text(value='./merged_model', description='Target Path:')
audit_btn = widgets.Button(description='Run Sanity Audit', button_style='info')
hub_token = widgets.Password(value='', description='HF Token:', placeholder='Optional if logged in')
hub_repo = widgets.Text(value='', description='HF Repo ID:', placeholder='username/model-name')
hub_private = widgets.Checkbox(value=False, description='Private Repo')
hub_btn = widgets.Button(description='Publish to Hub', button_style='primary')

tab_ops = widgets.VBox([audit_target, audit_btn, widgets.HTML('<hr/>'), hub_token, hub_repo, hub_private, hub_btn])

# Render Tab Container
studio_tabs = widgets.Tab()
studio_tabs.children = [tab_merge, tab_moe, tab_prune, tab_lora, tab_tok, tab_ops]
studio_tabs.set_title(0, 'Merge Models')
studio_tabs.set_title(1, 'MoE Upcycling')
studio_tabs.set_title(2, 'Pruning & Compression')
studio_tabs.set_title(3, 'LoRA Tools')
studio_tabs.set_title(4, 'Tokenizer Surgery')
studio_tabs.set_title(5, 'Audit & Publish')

output_panel = widgets.Output()

def on_merge_clicked(b):
    with output_panel:
        clear_output()
        import yaml
        cfg = {
            'merge_method': merge_method.value,
            'base_model': base_model_in.value.strip() or None,
            'models': [
                {'model': m1_in.value.strip(), 'weight': w1_in.value, 'density': d1_in.value},
                {'model': m2_in.value.strip(), 'weight': w2_in.value, 'density': d2_in.value},
            ],
            'ignore_vision': vision_check.value,
            'skip_patterns': ['mlp.gate_proj', 'mlp.up_proj', 'mlp.down_proj'] if skip_mlp_check.value else [],
            'dtype': 'bfloat16',
        }
        with open('colab_merge.yaml', 'w') as f:
            yaml.dump(cfg, f)
        print('Saved config colab_merge.yaml. Launching PolyMerge...')
        !python -m polymerge.cli colab_merge.yaml -o {merge_out_dir.value}

def on_moe_clicked(b):
    with output_panel:
        clear_output()
        experts = [line.strip() for line in moe_experts.value.splitlines() if line.strip()]
        cfg = {
            'base_model': moe_base.value.strip(),
            'models': experts,
            'target_format': moe_arch.value,
            'gate_mode': moe_gate.value,
            'shared_expert': moe_shared.value.strip() or None,
        }
        with open('colab_moe.yaml', 'w') as f:
            yaml.dump(cfg, f)
        print('Saved config colab_moe.yaml. Launching MoE Upcycling...')
        !python -m polymerge.cli colab_moe.yaml -o {moe_out_dir.value} --moe

def on_prune_clicked(b):
    with output_panel:
        clear_output()
        if prune_type.value == 'Dense Layer Pruning':
            !python -m polymerge.cli --prune-dense --base {prune_model.value.strip()} -o {prune_out_dir.value} --drop-fraction {prune_dense_frac.value}
        else:
            !python -m polymerge.cli --prune-moe --base {prune_model.value.strip()} -o {prune_out_dir.value} --target-experts {prune_moe_experts.value} --alpha 0.3

def on_lora_clicked(b):
    with output_panel:
        clear_output()
        if lora_action.value == 'Extract LoRA (SVD)':
            !python -m polymerge.cli --extract-lora --base {lora_base.value.strip()} --tuned {lora_tuned.value.strip()} -o {lora_out_dir.value} --rank {lora_rank.value}
        elif lora_action.value == 'Bake LoRA into Base':
            !python -m polymerge.cli --merge-lora --base {lora_base.value.strip()} --adapter {lora_adapter.value.strip()} -o {lora_out_dir.value}
        elif lora_action.value == 'Merge Multiple Adapters':
            !python -m polymerge.cli --merge-adapters --adapters {lora_adapter.value.strip()} {lora_tuned.value.strip()} -o {lora_out_dir.value}

def on_tok_clicked(b):
    with output_panel:
        clear_output()
        if tok_action.value == 'Align Vocabularies':
            !python -m polymerge.cli --align-vocab --base {tok_base.value.strip()} --models {tok_sources.value.strip()} -o {tok_out_dir.value}
        else:
            !python -m polymerge.cli --truncate-vocab {tok_truncate_n.value} --base {tok_base.value.strip()} -o {tok_out_dir.value}

def on_audit_clicked(b):
    with output_panel:
        clear_output()
        !python -m polymerge.cli --audit --base {audit_target.value.strip()}

def on_hub_clicked(b):
    with output_panel:
        clear_output()
        repo = hub_repo.value.strip()
        if not repo:
            print('Please specify HF Repo ID (e.g. username/my-model)!')
            return
        from polymerge.publisher import PublisherTool
        tok = hub_token.value.strip() or None
        PublisherTool.push_to_hub(audit_target.value.strip(), repo, private=hub_private.value, token=tok)
        print(f'Successfully published to https://huggingface.co/{repo}')

merge_btn.on_click(on_merge_clicked)
moe_btn.on_click(on_moe_clicked)
prune_btn.on_click(on_prune_clicked)
lora_btn.on_click(on_lora_clicked)
tok_btn.on_click(on_tok_clicked)
audit_btn.on_click(on_audit_clicked)
hub_btn.on_click(on_hub_clicked)

display(studio_tabs)
display(output_panel)